# DLP Detection Model — Training Pipeline


In [0]:
%pip install shap numpy==1.26.4 --quiet

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import  roc_auc_score, average_precision_score, classification_report, confusion_matrix
import pickle
import os
import shap

## 1. Load Data

In [0]:
df_features_clean = spark.table("workspace.default.dlp_features_clean")
df = df_features_clean.toPandas()

print(f"Total rows: {len(df)}")


## 2. Define Feature Columns

In [0]:
LABEL_COL = "Risk_Label"

# Identifiers which are excluded from model
ID_COLS = ["Action_ID", "AccountDisplayName", "ObjectName", "Target_Domain", "Timestamp"]

# Categorical: OrdinalEncoder
CATEGORICAL_COLS = [
    "ActionType",
    "file_extension",
    "Position",
]

# Numerical / binary: passed through as-is
NUMERICAL_COLS = [
    "is_high_risk_extension",
    "file_name_has_sensitive_keyword",
    "double_extension_check",
    "is_risky_target_domain",
    "is_first_time_domain",
    "is_after_hours",
    "day_of_week",
    "is_monday_or_friday",
    "is_high_risk_position",
    "user_upload_count_24h",
    "user_unique_domains_7d",
]

FEATURE_COLS = CATEGORICAL_COLS + NUMERICAL_COLS

X = df[FEATURE_COLS]
y = df[LABEL_COL]

print(f"Features: {len(FEATURE_COLS)}")
print(f"X shape: {X.shape}")

## 3. Train / Test Split

In [0]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")
print(f"Train label distribution:\n{y_train.value_counts()}")

## 4. Build Pipeline
OrdinalEncoder for categoricals (handles unseen values) → GradientBoostingClassifier

In [0]:
# OrdinalEncoder for categorical columns
# handle_unknown='use_encoded_value' + unknown_value=-1 handles unseen categories at inference
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ), CATEGORICAL_COLS),
        ("num", "passthrough", NUMERICAL_COLS),
    ]
)

gbt = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier",   gbt)
])

print("Pipeline defined.")

## 5. Train

In [0]:
pipeline.fit(X_train, y_train)
print("Done")

## 6. Evaluate
AUC-PR is the primary metric that focuses on precision/recall tradeoff
which matters more than accuracy in DLP.

In [0]:
y_pred      = pipeline.predict(X_test)
y_pred_prob = pipeline.predict_proba(X_test)[:, 1]

print(f"AUC-ROC  : {roc_auc_score(y_test, y_pred_prob):.4f}")
print(f"AUC-PR   : {average_precision_score(y_test, y_pred_prob):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

## 7. Feature Importance
Shows which features the model relied on most.

In [0]:
feature_names = CATEGORICAL_COLS + NUMERICAL_COLS

importance_df = (
    pd.DataFrame({
        "feature":    feature_names,
        "importance": pipeline.named_steps["classifier"].feature_importances_
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)
display(importance_df)

## 8. Save Model

In [0]:

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_root = os.path.dirname(notebook_path)

model_path = f"/Workspace{repo_root}/dlp_gbt_v1.pkl"
with open(model_path, "wb") as f:
    pickle.dump(pipeline, f)


## 9. Results Review


In [0]:
df_review = df.copy()
df_review["predicted"]      = pipeline.predict(df_review[FEATURE_COLS])
df_review["predicted_prob"] = pipeline.predict_proba(df_review[FEATURE_COLS])[:, 1]

test_indices = X_test.index
false_negatives = df_review.loc[test_indices][
    (df_review.loc[test_indices]["Risk_Label"] == 1) &
    (df_review.loc[test_indices]["predicted"] == 0)
]

print(f"False negatives: {len(false_negatives)}")
display(false_negatives.sort_values("predicted_prob", ascending=False))

In [0]:
action_id = 163

gbt_model = pipeline.named_steps["classifier"]

X_transformed = pipeline.named_steps["preprocessor"].transform(df_review[FEATURE_COLS])

explainer = shap.TreeExplainer(gbt_model)

row_idx = df_review [df_review["Action_ID"] == action_id].index[0]
row_transformed = X_transformed[df_review .index.get_loc(row_idx)]

shap_values = explainer.shap_values(row_transformed.reshape(1, -1))
shap_df = (
    pd.DataFrame({
        "feature":    FEATURE_COLS,
        "value":      df_review .loc[row_idx, FEATURE_COLS].values,
        "shap_value": shap_values[0]
    })
    .sort_values("shap_value", ascending=False)
    .reset_index(drop=True)
)
print(shap_df.to_string(index=False))

In [0]:
display(df_review[df_review["Action_ID"] == action_id])